# Load Test Data and Trained Model

In this section, the held-out test dataset and the best-performing model are loaded.  
Predictions are then generated on the test set, providing the basis for evaluating the model's performance on unseen data.

In [ ]:
import pandas as pd
import joblib

test_df = pd.read_parquet("../data/test.parquet")

X_test = test_df["processed_text"]
y_test = test_df["label_text"]

model = joblib.load("../model/best_model.pkl")

y_pred = model.predict(X_test)

## Test Set Evaluation

After selecting the optimal hyperparameters through cross-validation, the final Logistic Regression model is evaluated on the held-out test set. Since this dataset was not used during training or model selection, it provides an unbiased estimate of the model's ability to generalize to unseen customer requests.

The evaluation focuses on macro-averaged metrics, which assign equal importance to all intent classes regardless of their frequency. This is particularly appropriate for the Banking77 dataset, where class frequencies are not perfectly balanced.

## Results Interpretation

The model achieved a **Macro F1-score of 0.84** on the test set, compared with **0.86** during cross-validation. This small decrease indicates that the selected model generalizes well to unseen data and that the validation process provided a reliable estimate of real-world performance.

Although the training Macro F1-score reached **0.97**, the close agreement between the validation and test results suggests only **moderate overfitting**. Overall, the model maintains strong and consistent performance across unseen customer intents, demonstrating that the selected hyperparameters effectively balance model complexity and generalization.

In [16]:
from sklearn.metrics import accuracy_score, f1_score

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "F1-score (Macro)": f1_score(y_test, y_pred, average="macro"),
}

for name, value in metrics.items():
    print(f"{name:<20}: {value:.4f}")

Accuracy            : 0.8571
F1-score (Macro)    : 0.8551


## Per-Intent Performance Analysis

A detailed classification report is generated to evaluate the performance for each intent individually. Precision, Recall, F1-score, and Support are reported for every class, enabling identification of both strong and weak-performing intents.

In [20]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0
)

In [ ]:
report_df = pd.DataFrame(report).T
report_df.head()

,precision,recall,f1-score,support
Refund_not_showing_up,0.966667,0.906250,0.935484,32.0
activate_my_card,1.000000,0.968750,0.984127,32.0
age_limit,1.000000,0.954545,0.976744,22.0
apple_pay_or_google_pay,1.000000,0.960000,0.979592,25.0
atm_support,0.875000,0.823529,0.848485,17.0


## Best and Worst Performing Intents

To simplify interpretation, intents are ranked according to their F1-score. This highlights which customer intents are classified reliably and which remain challenging for the model.

In [28]:
best_labels = report_df.sort_values("f1-score", ascending=False).head(10)
best_labels

,precision,recall,f1-score,support
passcode_forgotten,1.000000,1.000000,1.000000,21.0
activate_my_card,1.000000,0.968750,0.984127,32.0
visa_or_mastercard,1.000000,0.962963,0.981132,27.0
card_about_to_expire,0.962963,1.000000,0.981132,26.0
verify_top_up,0.961538,1.000000,0.980392,25.0
apple_pay_or_google_pay,1.000000,0.960000,0.979592,25.0
age_limit,1.000000,0.954545,0.976744,22.0
getting_virtual_card,0.909091,1.000000,0.952381,20.0
top_up_limits,0.947368,0.947368,0.947368,19.0
get_disposable_virtual_card,0.947368,0.947368,0.947368,19.0


In [29]:
worst_labels = report_df.sort_values("f1-score").head(10)
worst_labels

,precision,recall,f1-score,support
top_up_failed,0.571429,0.551724,0.561404,29.0
why_verify_identity,0.857143,0.500000,0.631579,24.0
pending_top_up,0.678571,0.633333,0.655172,30.0
verify_my_identity,0.586207,0.809524,0.680000,21.0
card_delivery_estimate,0.666667,0.727273,0.695652,22.0
pending_transfer,0.740741,0.666667,0.701754,30.0
order_physical_card,0.633333,0.791667,0.703704,24.0
virtual_card_not_working,0.666667,0.750000,0.705882,8.0
card_acceptance,0.692308,0.750000,0.720000,12.0
lost_or_stolen_card,0.705882,0.750000,0.727273,16.0


## Misclassified Examples

The following table presents misclassified samples from the test set using the **preprocessed text** (`processed_text`) that was provided to the classifier, rather than the original raw customer queries. Since the model was trained and evaluated on the processed representation, these examples more accurately reflect the input used during prediction.

Reviewing these samples provides qualitative insight into the model's prediction errors and highlights common sources of confusion between semantically similar intents.

In [32]:
errors = pd.DataFrame({
    "Processed_Text": X_test,
    "True": y_test,
    "Predicted": y_pred
})

errors = errors[
    errors["True"] != errors["Predicted"]
]

errors.head(20)

,Processed_Text,True,Predicted
1,fee impose make transaction,card_payment_fee_charged,cancel_transfer
4,verification identity,why_verify_identity,verify_my_identity
7,withdrawal go smooth get decline today go,declined_cash_withdrawal,declined_transfer
9,need refund,request_refund,Refund_not_showing_up
13,reset darn pin number,pin_blocked,change_pin
23,not get app,lost_or_stolen_phone,unable_to_verify_identity
24,uknown charge show account,card_payment_not_recognised,transaction_charged_twice
30,extra fee get,cash_withdrawal_charge,card_payment_fee_charged
37,payment not transfer,failed_transfer,card_payment_not_recognised
42,need verify identity,unable_to_verify_identity,verify_my_identity


**Observation**

Most misclassified samples were assigned to intents that are semantically similar to the true labels. Common confusions occurred between related categories such as identity verification (`why_verify_identity` vs. `verify_my_identity`), PIN management (`pin_blocked` vs. `change_pin`), refund requests (`request_refund` vs. `refund_not_showing_up`), and transfer or payment issues.

These results indicate that the model generally captures the overall topic of a customer request but may struggle to distinguish between closely related banking intents, particularly when the input is short or lacks sufficient context.